# 04 — Pipeline GPU n°2 : PyTorch → torch.compile / TorchScript → CUDA → GPU

| Couche | Technologie |
|---|---|
| Haut niveau | PyTorch, ResNet-18 + tête 10 classes |
| Format du modèle | Graphe PyTorch 2 / Inductor / TorchScript |
| Middle level | Compilation et fusion d'opérateurs par PyTorch |
| Bas niveau | CUDA / cuDNN et kernels générés par Triton/Inductor |
| Matériel | GPU NVIDIA |

Le notebook mesure TorchScript CUDA (contrôle) puis `torch.compile` (Inductor +
Triton) sur le même modèle et les mêmes entrées.

In [ ]:
%matplotlib inline
import json
import os
import sys
import time
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "src" / "common.py").exists():
            return candidate
    return p

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TP_ROOT", str(ROOT))
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("TP_REPO_URL", "")
if IN_COLAB and REPO_URL and not (ROOT / "src" / "common.py").exists():
    import subprocess
    subprocess.run(["git", "clone", REPO_URL], check=True)
    ROOT = Path(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")).resolve()
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    os.environ["TP_ROOT"] = str(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from src import common as C

C.ensure_dirs()
C.set_seed()
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "4")))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Racine du projet :", ROOT)
print("PyTorch :", torch.__version__, "| CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print(f"Protocole : {C.N_IMAGES} images | batch={C.BATCH_SIZE} | {C.REPETITIONS} répétitions | warmup={C.WARMUP}")

In [ ]:
def show(result_or_id):
    result = C.load_result(result_or_id) if isinstance(result_or_id, str) else result_or_id
    print(f"{result['label']}  |  device={result['device']}  |  backend={result['backend']}")
    display(pd.DataFrame(C.summary_rows(result)).round(4))
    print(f"Top-1 accuracy = {result['accuracy_top1'] * 100:.2f} %")

def bar_metric(ids, metric, std_metric, ylabel, title, color="#4C72B0"):
    df = pd.DataFrame(C.comparison_rows(ids))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["pipeline"], df[metric], yerr=df[std_metric], capsize=5, color=color)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=18)
    for i, value in enumerate(df[metric]):
        if value is not None:
            ax.text(i, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    return df

## 1. Données et modèle

In [ ]:
images, labels = C.load_subset()
x_cpu = C.preprocess_images(images, device="cpu")
print("images :", images.shape, images.dtype)
print("tenseur prétraité :", tuple(x_cpu.shape), x_cpu.dtype)
print("distribution :", np.bincount(labels, minlength=len(C.CLASSES)).tolist())
print("classes :", C.CLASSES)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    ax.imshow(images[idx])
    ax.set_title(C.CLASSES[int(labels[idx])])
    ax.axis("off")
fig.suptitle("Exemples du sous-ensemble CIFAR-10 (5 premières classes)")
plt.tight_layout()
plt.show()

In [ ]:
model = C.build_model("cpu")
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 + tête 10 classes : {n_params/1e6:.2f} M paramètres")
print("Couche de classification :", model.fc)
print("Poids de la tête :", C.ARTIFACTS / "classifier_head.npz")

## 2. Vérification du GPU

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU requis. En local : docker compose run --rm tp ... (avec --gpus all). "
        "Sur Colab : Exécution > Modifier le type d'exécution > GPU."
    )
print("GPU :", torch.cuda.get_device_name(0))
print("CUDA :", torch.version.cuda, "| cuDNN :", torch.backends.cudnn.version())

## 3. TorchScript CUDA (contrôle)

In [ ]:
model_gpu = C.build_model("cuda")
example = torch.randn(C.BATCH_SIZE, 3, C.IMAGE_SIZE, C.IMAGE_SIZE, device="cuda")
traced = torch.jit.trace(model_gpu, example)
traced = torch.jit.freeze(traced.eval())
traced = torch.jit.optimize_for_inference(traced)
ts_path = C.ARTIFACTS / "resnet18_cifar10_torchscript_gpu.pt"
torch.jit.save(traced, ts_path)
print("TorchScript GPU :", ts_path, f"({ts_path.stat().st_size/1e6:.2f} Mo)")

fn_ts = C.make_torch_fn(traced, "cuda")
records, summary, logits_ts = C.run_benchmark(fn_ts, x_cpu, "cuda")
C.save_result(
    "04a_gpu_torchscript", records, summary, logits_ts, labels,
    device="cuda", backend="torchscript-cuda",
    notes="TorchScript CUDA, contrôle.",
)
show("04a_gpu_torchscript")

## 4. torch.compile (Inductor + Triton)

`torch.compile` capture le graphe, le compile avec Inductor et génère des
kernels Triton pour le GPU. La première exécution compile ; les warmups
absorbent ce coût avant les mesures.

In [ ]:
compiled = torch.compile(model_gpu, mode="default", fullgraph=False, dynamic=False)
fn_compiled = C.make_torch_fn(compiled, "cuda")
records, summary, logits_compiled = C.run_benchmark(fn_compiled, x_cpu, "cuda")
C.save_result(
    "04_gpu_torch_compile", records, summary, logits_compiled, labels,
    device="cuda", backend="torch.compile/inductor-cuda",
    notes="torch.compile mode=default, Inductor + Triton, FP32.",
)
show("04_gpu_torch_compile")

## 5. Comparaison et intégrité des prédictions

In [ ]:
ref = np.load(C.RESULTS / "00_reference_pytorch_gpu_logits.npy")
for rid in ["04a_gpu_torchscript", "04_gpu_torch_compile"]:
    r = C.load_result(rid)
    logits_r = np.load(C.RESULTS / f"{rid}_logits.npy")
    print(
        f"{r['label']:<36} Top-1={r['accuracy_top1']*100:5.2f} %  "
        f"accord={C.prediction_agreement(logits_r, ref)*100:5.2f} %"
    )
ids = ["00_reference_pytorch_gpu", "04a_gpu_torchscript", "04_gpu_torch_compile"]
display(pd.DataFrame(C.comparison_rows(ids)).round(4))
bar_metric(ids, "throughput_ips", "throughput_std", "images/s",
           "Pipeline GPU n°2 : débit", color="#55A868")
bar_metric(ids, "latency_ms", "latency_std", "ms / image",
           "Pipeline GPU n°2 : latence", color="#C44E52")

### Interprétation

`torch.compile` peut fusionner des opérateurs et réduire les lancements de
kernels, ce qui améliore souvent le débit sur GPU. Le gain dépend de la taille
de batch, du modèle et du temps de compilation. L'accord de prédictions avec la
référence GPU vérifie que les kernels générés conservent les décisions du
modèle. La comparaison avec TensorRT (notebook 03) montre deux philosophies :
un runtime externe spécialisé vs une compilation intégrée à PyTorch.